# Assignment 3 – CNN Image Classification using CIFAR-10
**For Jupyter Notebook (offline/local dataset preferred)**

Before starting:
1. Keep this notebook in your `DL_Practicals` folder.
2. If your college gives you an extracted folder named `cifar-10-batches-py`, keep that folder beside this notebook.
3. Run the cells from top to bottom using **Shift + Enter**.

The notebook first looks for the local CIFAR-10 Python batch folder. If it is not found, it tries the Keras CIFAR-10 dataset cache.

In [ ]:
# Cell 1: Import required libraries
import os
import pickle
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

print("Libraries imported successfully!")

In [ ]:
# Cell 2: Load CIFAR-10 from the local folder if available.
# Expected folder: cifar-10-batches-py
# Keep it in the same folder as this notebook.

dataset_folder = "cifar-10-batches-py"

def load_cifar_batch(file_path):
    with open(file_path, "rb") as f:
        batch = pickle.load(f, encoding="bytes")
    images = batch[b"data"]
    labels = batch[b"labels"]
    images = images.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return images, np.array(labels)

if os.path.isdir(dataset_folder):
    print("Local CIFAR-10 folder found. Loading files...")

    train_images = []
    train_labels = []

    for i in range(1, 6):
        file_path = os.path.join(dataset_folder, "data_batch_" + str(i))
        images, labels = load_cifar_batch(file_path)
        train_images.append(images)
        train_labels.extend(labels)

    x_train = np.concatenate(train_images, axis=0)
    y_train = np.array(train_labels)

    test_file = os.path.join(dataset_folder, "test_batch")
    x_test, y_test = load_cifar_batch(test_file)

    print("Loaded CIFAR-10 from the local folder.")

else:
    print("Local folder not found. Trying the Keras CIFAR-10 cache...")
    print("If the dataset is not cached, this step may require internet.")
    from tensorflow.keras.datasets import cifar10
    (x_train, y_train), (x_test, y_test) = cifar10.load_data()
    y_train = y_train.reshape(-1)
    y_test = y_test.reshape(-1)

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
# Cell 3: Normalize image pixel values
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Image normalization completed.")

In [ ]:
# Cell 4: Convert labels into one-hot categorical format
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("Label conversion completed.")
print("Training label shape:", y_train.shape)

In [ ]:
# Cell 5: Define the CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", padding="same",
           input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation="relu"),
    Dense(10, activation="softmax")
])

model.summary()

In [ ]:
# Cell 6: Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

In [ ]:
# Cell 7: Train the model
history = model.fit(
    x_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test)
)

In [ ]:
# Cell 8: Evaluate the model
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=2)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy * 100, "%")

In [ ]:
# Cell 9: Predict one test image
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

index = 0
prediction = model.predict(np.expand_dims(x_test[index], axis=0))
predicted_class = np.argmax(prediction[0])
actual_class = np.argmax(y_test[index])

plt.imshow(x_test[index])
plt.axis("off")
plt.show()

print("Predicted class:", class_names[predicted_class])
print("Actual class:", class_names[actual_class])

In [ ]:
# Cell 10: Plot training and validation accuracy
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("CNN Model Accuracy")
plt.legend()
plt.show()

In [ ]:
# Cell 11: Plot training and validation loss
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("CNN Model Loss")
plt.legend()
plt.show()

## Conclusion
Thus, we implemented image classification using a Convolutional Neural Network (CNN) on the CIFAR-10 dataset. The model was trained and evaluated using accuracy and loss, and it was used to predict the class of a test image.

**Note:** TensorFlow and Matplotlib must already be installed in the computer's Jupyter Python environment. For offline use, the extracted `cifar-10-batches-py` folder must be copied alongside this notebook.